# ML Preprocessing


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

df_clean = pd.read_csv('../data/processed/df_clean.csv', parse_dates=['Date Local'])
# Recreate the station category used by the original EDA/ML workflow.
df_clean['area_type'] = df_clean['City'].apply(
    lambda c: 'Rural/Non-city' if c == 'Not in a city' else 'Urban'
)
df_ml = df_clean.copy()
print('Clean dataset shape:', df_clean.shape)
print('ML dataset shape:', df_ml.shape)
df_clean.head()

Clean dataset shape: (70922, 28)
ML dataset shape: (70922, 28)


,station,State,City,Date Local,State Code,County Code,Site Num,NO2 Mean,NO2 1st Max Value,NO2 1st Max Hour,...,SO2 AQI,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI,Year,Month,Season,Region,area_type
0,11_1_41,District Of Columbia,Washington,2010-01-01,11,1,41,19.208333,30.0,4,...,7,0.962939,1.25,6,14.0,2010,1,Winter,South,Urban
1,11_1_41,District Of Columbia,Washington,2010-01-02,11,1,41,8.708333,10.0,0,...,6,0.772917,0.80,6,9.0,2010,1,Winter,South,Urban
2,11_1_41,District Of Columbia,Washington,2010-01-03,11,1,41,7.291667,10.0,17,...,6,0.827083,0.90,18,10.0,2010,1,Winter,South,Urban
3,11_1_41,District Of Columbia,Washington,2010-01-04,11,1,41,13.041667,22.0,8,...,9,0.906250,0.95,4,10.0,2010,1,Winter,South,Urban
4,11_1_41,District Of Columbia,Washington,2010-01-05,11,1,41,15.173913,22.0,7,...,11,0.925947,1.00,14,11.0,2010,1,Winter,South,Urban


In [2]:
print(df_ml.shape)

(70922, 28)


In [3]:
df_ml = df_ml.sort_values(['station', 'Date Local']).reset_index(drop=True)

In [4]:
static_cols = ['State', 'City', 'State Code', 'County Code', 'Site Num',
               'Region', 'area_type']
static = df_ml.drop_duplicates('station').set_index('station')[static_cols]


In [5]:
def fill_calendar(g):
    st = g['station'].iloc[0] 
    full = pd.date_range(g['Date Local'].min(), g['Date Local'].max(), freq='D')
    out = g.drop(columns='station').set_index('Date Local').reindex(full)
    out.index.name = 'Date Local'
    out['station'] = st
    return out.reset_index()

In [6]:
before = len(df_ml)
df_ml = pd.concat([fill_calendar(g) for _, g in df_ml.groupby('station')],
                  ignore_index=True)


df_ml['was_missing'] = df_ml['O3 AQI'].isna()
for c in static_cols:
    df_ml[c] = df_ml['station'].map(static[c])
df_ml['Year'] = df_ml['Date Local'].dt.year
df_ml['Month'] = df_ml['Date Local'].dt.month
df_ml['Season'] = df_ml['Month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall', 11: 'Fall'})

print("Before : ", before, " After : ", len(df_ml), " added : ", len(df_ml) - before)
print(df_ml['was_missing'].sum())

Before :  70922  After :  75511  added :  4589
4589


In [7]:
print(len(df_ml))                       
print(df_ml['was_missing'].sum())       
print(df_ml.groupby('station')['Date Local'].diff().dt.days.max())  

75511
4589
1.0


In [8]:
# Causal missing-value treatment: no future observations used.
# Preserve the original O3 AQI target; only historical inputs may be filled.
df_ml = df_ml.sort_values(['station', 'Date Local']).reset_index(drop=True)
df_ml['O3_AQI_history'] = df_ml['O3 AQI']

for col in ['O3_AQI_history', 'NO2 AQI', 'SO2 AQI', 'CO AQI']:
    df_ml[col] = df_ml.groupby('station')[col].ffill(limit=7)

print(df_ml[['O3 AQI', 'O3_AQI_history', 'NO2 AQI', 'SO2 AQI', 'CO AQI']].isna().sum())

O3 AQI            4589
O3_AQI_history    2287
NO2 AQI           2287
SO2 AQI           2287
CO AQI            2290
dtype: int64


In [9]:
# Mark only historical O3 values filled from earlier observations.
df_ml['was_filled_from_past'] = df_ml['O3 AQI'].isna() & df_ml['O3_AQI_history'].notna()
print('O3 historical values filled from past:', df_ml['was_filled_from_past'].sum())

O3 historical values filled from past: 2302


In [10]:
df_ml = df_ml.sort_values(['station', 'Date Local']).reset_index(drop=True)
for lag in [1, 7, 30]:
    df_ml[f'O3_AQI_lag{lag}'] = df_ml.groupby('station')['O3_AQI_history'].shift(lag)
print(df_ml[['station', 'Date Local', 'O3 AQI', 'O3_AQI_lag1', 'O3_AQI_lag7', 'O3_AQI_lag30']].head(35))

    station Date Local  O3 AQI  O3_AQI_lag1  O3_AQI_lag7  O3_AQI_lag30
0   11_1_41 2010-01-01    24.0          NaN          NaN           NaN
1   11_1_41 2010-01-02    29.0         24.0          NaN           NaN
2   11_1_41 2010-01-03    29.0         29.0          NaN           NaN
3   11_1_41 2010-01-04    21.0         29.0          NaN           NaN
4   11_1_41 2010-01-05    22.0         21.0          NaN           NaN
5   11_1_41 2010-01-06    19.0         22.0          NaN           NaN
6   11_1_41 2010-01-07    19.0         19.0          NaN           NaN
7   11_1_41 2010-01-08    19.0         19.0         24.0           NaN
8   11_1_41 2010-01-09    28.0         19.0         29.0           NaN
9   11_1_41 2010-01-10    26.0         28.0         29.0           NaN
10  11_1_41 2010-01-11    19.0         26.0         21.0           NaN
11  11_1_41 2010-01-12    17.0         19.0         22.0           NaN
12  11_1_41 2010-01-13    10.0         17.0         19.0           NaN
13  11

In [11]:
g = df_ml.groupby('station')['O3_AQI_history']
df_ml['O3_roll_mean7'] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=5).mean())
df_ml['O3_roll_std7'] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=5).std())

In [12]:
print(df_ml[['station', 'Date Local', 'O3 AQI', 'O3_roll_mean7', 'O3_roll_std7']].iloc[5:15])

    station Date Local  O3 AQI  O3_roll_mean7  O3_roll_std7
5   11_1_41 2010-01-06    19.0      25.000000      3.807887
6   11_1_41 2010-01-07    19.0      24.000000      4.195235
7   11_1_41 2010-01-08    19.0      23.285714      4.270608
8   11_1_41 2010-01-09    28.0      22.571429      4.540820
9   11_1_41 2010-01-10    26.0      22.428571      4.314979
10  11_1_41 2010-01-11    19.0      22.000000      3.651484
11  11_1_41 2010-01-12    17.0      21.714286      3.817254
12  11_1_41 2010-01-13    10.0      21.000000      4.203173
13  11_1_41 2010-01-14     7.0      19.714286      5.936168
14  11_1_41 2010-01-15     8.0      18.000000      7.659417


In [13]:
df_ml = df_ml.sort_values(['station', 'Date Local']).reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']

for p in pollutants:
    g = df_ml.groupby('station')[f'{p} AQI']

    
    for lag in [1, 7, 30]:
        df_ml[f'{p}_AQI_lag{lag}'] = g.shift(lag)

    
    df_ml[f'{p}_roll_mean7'] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=5).mean())
    df_ml[f'{p}_roll_std7']  = g.transform(lambda s: s.shift(1).rolling(7, min_periods=5).std())

    

In [14]:
print(df_ml.filter(regex='lag|roll').shape[1])

20


In [15]:
print(df_ml[['station', 'Date Local', 'NO2 AQI', 'NO2_AQI_lag1', 'NO2_roll_mean7']].iloc[5:10])

   station Date Local  NO2 AQI  NO2_AQI_lag1  NO2_roll_mean7
5  11_1_41 2010-01-06     22.0          21.0       17.600000
6  11_1_41 2010-01-07     35.0          22.0       18.333333
7  11_1_41 2010-01-08     31.0          35.0       20.714286
8  11_1_41 2010-01-09     30.0          31.0       21.142857
9  11_1_41 2010-01-10     25.0          30.0       24.142857


In [16]:
target = 'O3 AQI'

lag_roll = [c for c in df_ml.columns if 'lag' in c or 'roll' in c]
calendar = ['Month', 'Year']              
meta = ['station', 'Date Local', 'Region', 'Season', 'area_type']

features = lag_roll + calendar
print(len(features), features[:5])

22 ['O3_AQI_lag1', 'O3_AQI_lag7', 'O3_AQI_lag30', 'O3_roll_mean7', 'O3_roll_std7']


In [17]:
# Require an observed target and complete historical features.
df_model = df_ml.dropna(subset=features + [target]).copy()
print('Final ML Dataset:', df_model.shape)
print('Missing feature values:', df_model[features].isna().sum().sum())
print('Missing target values:', df_model[target].isna().sum())

Final ML Dataset: (67991, 51)
Missing feature values: 0
Missing target values: 0


In [18]:
train = df_model[df_model['Date Local'] <  '2014-01-01']   # 2010-2013
val   = df_model[(df_model['Date Local'] >= '2014-01-01') &
                 (df_model['Date Local'] <  '2015-01-01')] # 2014
test  = df_model[df_model['Date Local'] >= '2015-01-01']   # 2015

print(len(train), len(val), len(test))
print(train['Date Local'].max(), val['Date Local'].min(), val['Date Local'].max(), test['Date Local'].min())

44847 12010 11134
2013-12-31 00:00:00 2014-01-01 00:00:00 2014-12-31 00:00:00 2015-01-01 00:00:00


In [19]:
# Split integrity checks
assert len(train) > 0 and len(val) > 0 and len(test) > 0, 'One of the splits is empty.'
assert train['Date Local'].max() < val['Date Local'].min()
assert val['Date Local'].max() < test['Date Local'].min()
for split in (train, val, test):
    assert not split[features + [target]].isna().any().any()
print('Chronological split verified; no missing model inputs or targets.')

Chronological split verified; no missing model inputs or targets.


In [20]:
X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]

In [21]:
print(X_train.shape, X_val.shape, X_test.shape)
print(X_train.isna().sum().sort_values(ascending=False).head(5))

(44847, 22) (12010, 22) (11134, 22)
O3_AQI_lag1      0
O3_AQI_lag7      0
Month            0
CO_roll_std7     0
CO_roll_mean7    0
dtype: int64


In [22]:
df_model

,Date Local,State,City,State Code,County Code,Site Num,NO2 Mean,NO2 1st Max Value,NO2 1st Max Hour,NO2 AQI,...,SO2_AQI_lag1,SO2_AQI_lag7,SO2_AQI_lag30,SO2_roll_mean7,SO2_roll_std7,CO_AQI_lag1,CO_AQI_lag7,CO_AQI_lag30,CO_roll_mean7,CO_roll_std7
30,2010-01-31,District Of Columbia,Washington,11,1,41,24.458333,49.0,20.0,46.0,...,14.0,4.0,7.0,8.714286,5.023753,16.0,16.0,14.0,16.142857,2.853569
31,2010-02-01,District Of Columbia,Washington,11,1,41,35.958333,57.0,21.0,54.0,...,14.0,4.0,6.0,10.142857,4.879500,18.0,14.0,9.0,16.428571,2.935821
32,2010-02-02,District Of Columbia,Washington,11,1,41,40.909091,58.0,6.0,55.0,...,11.0,6.0,6.0,11.142857,4.059087,24.0,14.0,10.0,17.857143,3.848314
33,2010-02-03,District Of Columbia,Washington,11,1,41,30.791667,42.0,0.0,40.0,...,11.0,9.0,9.0,11.857143,3.387653,28.0,17.0,10.0,19.857143,4.980916
34,2010-02-04,District Of Columbia,Washington,11,1,41,27.791667,50.0,23.0,47.0,...,10.0,7.0,11.0,12.000000,3.265986,5.0,22.0,11.0,18.142857,7.537209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75506,2015-12-27,Connecticut,New Haven,9,9,27,8.300000,18.3,17.0,17.0,...,1.0,1.0,1.0,0.428571,0.534522,5.0,5.0,11.0,5.857143,1.463850
75507,2015-12-28,Connecticut,New Haven,9,9,27,5.650000,11.3,16.0,10.0,...,0.0,1.0,1.0,0.285714,0.487950,3.0,5.0,6.0,5.571429,1.812654
75508,2015-12-29,Connecticut,New Haven,9,9,27,11.108333,18.8,8.0,17.0,...,0.0,0.0,0.0,0.142857,0.377964,2.0,5.0,2.0,5.142857,2.267787
75509,2015-12-30,Connecticut,New Haven,9,9,27,17.150000,23.2,18.0,22.0,...,0.0,0.0,0.0,0.142857,0.377964,3.0,8.0,5.0,4.857143,2.410295


In [23]:
cat_cols = ['Region', 'area_type', 'Season']
num_cols = [c for c in features if c != 'Month']

In [24]:
train, val, test = train.copy(), val.copy(), test.copy()


In [25]:
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(train[cat_cols])
le_cols = [c + '_le' for c in cat_cols]

for d in (train, val, test):
    d[le_cols] = oe.transform(d[cat_cols])

raw_cols = features + le_cols
X_train, X_val, X_test = train[raw_cols], val[raw_cols], test[raw_cols]

In [26]:
sc = StandardScaler().fit(train[num_cols])

def make_std(d):
    num = pd.DataFrame(sc.transform(d[num_cols]), columns=num_cols, index=d.index)
    return pd.concat([num, d[['Month']], d[le_cols]], axis=1)

X_train_std, X_val_std, X_test_std = make_std(train), make_std(val), make_std(test)
print(X_train.shape, X_train_std.shape)

(44847, 25) (44847, 25)


In [27]:
print("X_train columns:", X_train.shape[1])
for i, c in enumerate(X_train.columns, 1):
    print(f"{i:2d}. {c}")

X_train columns: 25
 1. O3_AQI_lag1
 2. O3_AQI_lag7
 3. O3_AQI_lag30
 4. O3_roll_mean7
 5. O3_roll_std7
 6. NO2_AQI_lag1
 7. NO2_AQI_lag7
 8. NO2_AQI_lag30
 9. NO2_roll_mean7
10. NO2_roll_std7
11. SO2_AQI_lag1
12. SO2_AQI_lag7
13. SO2_AQI_lag30
14. SO2_roll_mean7
15. SO2_roll_std7
16. CO_AQI_lag1
17. CO_AQI_lag7
18. CO_AQI_lag30
19. CO_roll_mean7
20. CO_roll_std7
21. Month
22. Year
23. Region_le
24. area_type_le
25. Season_le


In [28]:
print("\nX_train_std columns:", X_train_std.shape[1])
for i, c in enumerate(X_train_std.columns, 1):
    print(f"{i:2d}. {c}")


X_train_std columns: 25
 1. O3_AQI_lag1
 2. O3_AQI_lag7
 3. O3_AQI_lag30
 4. O3_roll_mean7
 5. O3_roll_std7
 6. NO2_AQI_lag1
 7. NO2_AQI_lag7
 8. NO2_AQI_lag30
 9. NO2_roll_mean7
10. NO2_roll_std7
11. SO2_AQI_lag1
12. SO2_AQI_lag7
13. SO2_AQI_lag30
14. SO2_roll_mean7
15. SO2_roll_std7
16. CO_AQI_lag1
17. CO_AQI_lag7
18. CO_AQI_lag30
19. CO_roll_mean7
20. CO_roll_std7
21. Year
22. Month
23. Region_le
24. area_type_le
25. Season_le




The outputs `X_train_std`, `X_val_std`, `X_test_std`, `y_train`, `y_val`, and `y_test` are prepared for model training and evaluation in the next notebook.

## Save prepared data for Supervised Learning



In [30]:
from pathlib import Path

output_dir = Path('../data/final')
output_dir.mkdir(parents=True, exist_ok=True)

# Save features
X_train_std.to_csv(output_dir / 'X_train.csv', index=False)
X_val_std.to_csv(output_dir / 'X_val.csv', index=False)
X_test_std.to_csv(output_dir / 'X_test.csv', index=False)

# Save target
y_train.to_csv(output_dir / 'y_train.csv', index=False, header=['O3_AQI'])
y_val.to_csv(output_dir / 'y_val.csv', index=False, header=['O3_AQI'])
y_test.to_csv(output_dir / 'y_test.csv', index=False, header=['O3_AQI'])

print("All datasets saved successfully!")

All datasets saved successfully!
